In [0]:
-- 1. General account statistics (games, winrate, avg K/D/A, duration)
WITH m AS (
    SELECT
        *,
        CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT) AS is_win
    FROM workspace.default.matches
)
SELECT
    COUNT(*)                        AS games,
    ROUND(100 * AVG(is_win), 2)     AS winrate,
    ROUND(AVG(kills), 2)            AS avg_kills,
    ROUND(AVG(deaths), 2)           AS avg_deaths,
    ROUND(AVG(assists), 2)          AS avg_assists,
    ROUND(AVG(duration) / 60.0, 2)  AS avg_duration_min
FROM m;

games,winrate,avg_kills,avg_deaths,avg_assists,avg_duration_min
100,52.0,4.56,9.46,15.63,39.23


In [0]:
-- 2. Statistics per side (matches, wins, winrate)
WITH m AS (
    SELECT
        *,
        CASE WHEN player_slot < 128 THEN 'radiant' ELSE 'dire' END AS side,
        CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT) AS is_win
    FROM workspace.default.matches
)
SELECT
    side,
    COUNT(*)                     AS matches,
    SUM(is_win)                  AS wins,
    ROUND(100 * AVG(is_win), 2)  AS winrate
FROM m
GROUP BY side
ORDER BY side DESC;

side,matches,wins,winrate
radiant,55,33,60.0
dire,45,19,42.22


Databricks visualization. Run in Databricks to view.

In [0]:
-- 3. Statistics per hero (min 3 games)
WITH m AS (
    SELECT
        *,
        CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT) AS is_win,
        (kills + assists) / GREATEST(deaths, 1) AS kda
    FROM workspace.default.matches
)
SELECT
    h.localized_name                AS hero,
    COUNT(*)                        AS games,
    SUM(m.is_win)                   AS wins,
    ROUND(100 * AVG(m.is_win), 2)   AS winrate,
    ROUND(AVG(m.kills), 2)          AS avg_kills,
    ROUND(AVG(m.deaths), 2)         AS avg_deaths,
    ROUND(AVG(m.assists), 2)        AS avg_assists,
    ROUND(AVG(m.kda), 2)            AS avg_kda
FROM m
JOIN workspace.default.heroes h ON m.hero_id = h.id
GROUP BY h.id, h.localized_name
HAVING COUNT(*) >= 3
ORDER BY hero;

hero,games,wins,winrate,avg_kills,avg_deaths,avg_assists,avg_kda
Bane,49,33,67.35,3.8,9.69,18.49,2.88
Bounty Hunter,3,3,100.0,5.67,6.0,26.0,5.88
Death Prophet,3,0,0.0,7.67,9.67,11.33,1.97
Lion,5,3,60.0,2.8,9.0,12.2,1.87
Luna,4,2,50.0,7.75,5.5,10.5,3.48
Witch Doctor,10,5,50.0,5.2,8.8,12.7,2.57


Databricks visualization. Run in Databricks to view.

In [0]:
-- 4. Stats for each match (newest first)
SELECT
    match_id,
    to_date(from_unixtime(start_time))                                AS day,
    (player_slot < 128) = CAST(radiant_win AS BOOLEAN)                AS win,
    ROUND((kills + assists) / GREATEST(deaths, 1), 2)                 AS kda,
    ROUND(duration / 60.0, 2)                                         AS duration_min
FROM workspace.default.matches
ORDER BY start_time DESC;

match_id,day,win,kda,duration_min
9048444944,2026-10-17,false,1.58,41.78
9048413977,2026-10-17,true,2.44,45.55
9046946594,2026-10-16,false,1.27,60.77
9046923400,2026-10-16,false,0.78,28.92
9046904626,2026-10-16,false,1.0,29.83
9046887347,2026-10-15,true,1.83,25.78
9043908501,2026-10-13,true,4.17,36.23
9042496006,2026-10-13,false,4.0,48.42
9042414408,2026-10-12,false,1.0,29.58
9042380320,2026-10-12,true,2.88,35.62


In [0]:
-- 5. Statistics per week (ISO week, same as %G-%V in SQLite)
WITH m AS (
    SELECT
        *,
        to_date(from_unixtime(start_time)) AS day,
        CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT) AS is_win,
        (kills + assists) / GREATEST(deaths, 1) AS kda
    FROM workspace.default.matches
)
SELECT
    concat(extract(YEAROFWEEK FROM day), '-', lpad(CAST(weekofyear(day) AS STRING), 2, '0')) AS week,
    COUNT(*)                     AS games,
    ROUND(100 * AVG(is_win), 2)  AS winrate,
    ROUND(AVG(kda), 2)           AS avg_kda
FROM m
GROUP BY 1
ORDER BY week;

week,games,winrate,avg_kda
2026-26,8,50.0,1.9
2026-27,7,14.29,1.09
2026-28,2,0.0,1.69
2026-29,4,50.0,2.27
2026-30,6,66.67,3.68
2026-31,2,0.0,1.73
2026-32,7,57.14,2.75
2026-33,5,20.0,1.98
2026-34,2,100.0,2.25
2026-36,15,66.67,3.66


Databricks visualization. Run in Databricks to view.

In [0]:
-- 6. Cumulative winrate (window function)
WITH m AS (
    SELECT
        *,
        CAST((player_slot < 128) = CAST(radiant_win AS BOOLEAN) AS INT) AS is_win
    FROM workspace.default.matches
)
SELECT
    to_date(from_unixtime(start_time))  AS day,
    is_win,
    SUM(is_win) OVER w                  AS wins_so_far,
    COUNT(*) OVER w                     AS games_so_far,
    ROUND(100.0 * SUM(is_win) OVER w / COUNT(*) OVER w, 2) AS cumulative_winrate
FROM m
WINDOW w AS (ORDER BY start_time ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
ORDER BY start_time DESC;

day,is_win,wins_so_far,games_so_far,cumulative_winrate
2026-10-17,0,52,100,52.00
2026-10-17,1,52,99,52.53
2026-10-16,0,51,98,52.04
2026-10-16,0,51,97,52.58
2026-10-16,0,51,96,53.13
2026-10-15,1,51,95,53.68
2026-10-13,1,50,94,53.19
2026-10-13,0,49,93,52.69
2026-10-12,0,49,92,53.26
2026-10-12,1,49,91,53.85


Databricks visualization. Run in Databricks to view.